# 02 — MCP + ReAct: The Model Drives the Workflow

**Goal:** Run the same edge-case query (`ACC-002`) through a ReAct agent connected to  
**two independent MCP servers** — one internal, one simulating a third-party provider.

---

## What Is ReAct?

**ReAct** (Reasoning + Acting) is a prompting pattern where the model interleaves:  
- **THOUGHT** — internal reasoning about what to do next  
- **ACTION** — a tool call with arguments  
- **OBSERVATION** — the result the tool returns  

```
THOUGHT → ACTION → OBSERVATION → THOUGHT → ACTION → OBSERVATION → ... → FINAL ANSWER
```

### Why does this solve the API chain's problems?

| Problem from NB01 | ReAct solution |
|------------------|-----------------|
| Silently drops 2nd shipment | Model *reads* the list and *decides* to check both |
| Wrong severity labelling | Model reasons about severity from context |
| Misses origin weather | Model backtracks and calls `get_weather(origin)` |
| No cross-referencing | Model explicitly correlates carrier + weather data |
| No natural language explanation | Model generates the explanation itself |

---

## Two MCP Servers — The Real-World Architecture

This notebook connects to **two independent MCP servers**, mirroring production:

```
ReAct Agent (LLM client)
    │
    ├── logistics_mcp_server.py   ← YOUR internal system
    │     tools: get_account, get_shipments,
    │             get_carrier_status, assess_delivery_risk
    │
    └── weather_mcp_server.py     ← THIRD-PARTY provider (Tomorrow.io, etc.)
          tools: get_weather, get_weather_multi
```

The agent receives a **unified 6-tool catalogue** at runtime. It never knows — or  
needs to know — which server owns which tool. That's the protocol abstraction at work.

In a REST-only world you'd write a custom aggregation layer to stitch these two  
data sources together. With MCP, **the protocol is the integration layer**.

In [ ]:
import os, sys
os.chdir(os.path.abspath(".."))  # project root

import requests
from rich import print as rprint
from rich.panel import Panel
from rich.console import Console
from dotenv import load_dotenv

load_dotenv()  # load OPENAI_API_KEY / ANTHROPIC_API_KEY from .env

console = Console()

# Verify servers
assert requests.get("http://localhost:8001/").status_code == 200, "Logistics API not running!"
assert requests.get("http://localhost:8002/").status_code == 200, "Weather API not running!"
print("✅ Mock servers reachable.")

# LLM provider config
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openai")
print(f"LLM Provider: {LLM_PROVIDER}")

## Option A — Custom Lightweight ReAct Loop (No Framework)

This implementation shows the ReAct pattern from first principles.  
Each iteration: build prompt → call LLM → parse action → call tool → append observation.

> **Skip to Option B** if you prefer using LangChain's built-in ReAct agent.

In [ ]:
import json
import re
import time

# ──────────────────────────────────────────────────────────────────────────────
# Tool definitions — annotated by MCP server origin
#
# In a live MCP session, the client connects to both servers and discovers
# these tools automatically via the MCP handshake. Here we define them as
# plain Python functions backed by direct HTTP calls so the ReAct loop works
# without launching subprocess MCP servers.
#
# The server annotations below are the key architectural point:
#   SERVER A (logistics-internal) — tools you own
#   SERVER B (weather-provider)   — tools from a third-party provider
#
# The agent sees both sets in a single catalogue and doesn't know the boundary.
# ──────────────────────────────────────────────────────────────────────────────

LOGISTICS = "http://localhost:8001"
WEATHER   = "http://localhost:8002"

# ── SERVER A: logistics-internal ─────────────────────────────────────────────

def get_account(account_id: str) -> dict:
    """[logistics-internal] Look up customer account."""
    return requests.get(f"{LOGISTICS}/accounts/{account_id}").json()

def get_shipments(account_id: str) -> list:
    """[logistics-internal] Get ALL shipments for account. May return multiple."""
    return requests.get(f"{LOGISTICS}/shipments", params={"account_id": account_id}).json()

def get_carrier_status(carrier_id: str) -> dict:
    """[logistics-internal] Get carrier operational status."""
    return requests.get(f"{LOGISTICS}/carriers/{carrier_id}").json()

def assess_delivery_risk(shipment_id: str) -> dict:
    """[logistics-internal] Composite risk summary — calls carrier + weather internally."""
    ship    = requests.get(f"{LOGISTICS}/shipments/{shipment_id}").json()
    carrier = requests.get(f"{LOGISTICS}/carriers/{ship['carrier_id']}").json()
    orig_wx = requests.get(f"{WEATHER}/weather/{ship['origin_city']}").json()
    dest_wx = requests.get(f"{WEATHER}/weather/{ship['destination_city']}").json()
    return {"shipment": ship, "carrier": carrier, "origin_weather": orig_wx, "dest_weather": dest_wx}

# ── SERVER B: weather-provider ────────────────────────────────────────────────

def get_weather(city: str) -> dict:
    """[weather-provider] Weather conditions + risk_level for a city."""
    return requests.get(f"{WEATHER}/weather/{city}").json()

def get_weather_multi(cities: list) -> dict:
    """[weather-provider] Weather for multiple cities in one call."""
    return {city: requests.get(f"{WEATHER}/weather/{city}").json() for city in cities}

# ── Unified catalogue (agent sees this — server origin is invisible to it) ───

TOOL_REGISTRY = {
    # From logistics-internal
    "get_account":          get_account,
    "get_shipments":        get_shipments,
    "get_carrier_status":   get_carrier_status,
    "assess_delivery_risk": assess_delivery_risk,
    # From weather-provider
    "get_weather":          get_weather,
    "get_weather_multi":    get_weather_multi,
}

TOOL_DESCRIPTIONS = """
Available tools (aggregated from two MCP servers):

[logistics-internal — your system]
- get_account(account_id: str) -> account info (name, tier, home_city)
- get_shipments(account_id: str) -> list of ALL shipments (may be multiple!)
- get_carrier_status(carrier_id: str) -> carrier status (on_time/delayed/disrupted), delay_hours
- assess_delivery_risk(shipment_id: str) -> composite risk summary for one shipment

[weather-provider — third-party]
- get_weather(city: str) -> weather conditions + risk_level (low/medium/high)
- get_weather_multi(cities: list) -> weather for multiple cities at once
"""

print("✅ Tool registry ready:", list(TOOL_REGISTRY.keys()))
print("   4 tools from logistics-internal, 2 tools from weather-provider")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ReAct System Prompt
# ──────────────────────────────────────────────────────────────────────────────

REACT_SYSTEM_PROMPT = f"""You are a shipment risk advisor. Answer user questions by using the available tools.

{TOOL_DESCRIPTIONS}

Use the following format for EVERY response until you have a final answer:

Thought: [your reasoning about what to do next]
Action: tool_name(arg1="value1", arg2="value2")
Observation: [tool result - this will be filled in by the system]

Repeat Thought/Action/Observation as many times as needed.

When you have enough information, respond with:
Final Answer: [your complete natural language risk assessment]

IMPORTANT:
- Accounts can have MULTIPLE shipments. Always process ALL of them.
- Check weather for BOTH origin and destination of each shipment.
- Cross-reference carrier status with weather risk.
- Be explicit about your confidence and reasoning.
"""

print("System prompt ready.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ReAct Loop Implementation — with token + cost tracking
# ──────────────────────────────────────────────────────────────────────────────

# Approximate pricing per 1M tokens (March 2026 estimates — verify before publishing)
TOKEN_PRICING = {
    "openai": {
        "model":  "gpt-4o",
        "input":  2.50,   # $/1M input tokens
        "output": 10.00,  # $/1M output tokens
    },
    "anthropic": {
        "model":  "claude-3-5-sonnet-20241022",
        "input":  3.00,   # $/1M input tokens
        "output": 15.00,  # $/1M output tokens
    },
}

def estimate_cost(prompt_tokens: int, completion_tokens: int, provider: str) -> float:
    """Calculate estimated USD cost from token counts."""
    pricing = TOKEN_PRICING[provider]
    return round(
        (prompt_tokens   / 1_000_000) * pricing["input"] +
        (completion_tokens / 1_000_000) * pricing["output"],
        6
    )


def call_llm(messages: list, provider: str = LLM_PROVIDER) -> tuple[str, dict]:
    """
    Call the LLM and return (response_text, usage_dict).
    usage_dict keys: prompt_tokens, completion_tokens, total_tokens
    """
    if provider == "openai":
        from openai import OpenAI
        client   = OpenAI()
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0,
            stop=["Observation:"]
        )
        text  = response.choices[0].message.content
        usage = {
            "prompt_tokens":     response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens":      response.usage.total_tokens,
        }
    elif provider == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = messages[0]["content"] if messages[0]["role"] == "system" else ""
        conv   = [m for m in messages if m["role"] != "system"]
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=2048,
            system=system,
            messages=conv,
            stop_sequences=["Observation:"]
        )
        text  = response.content[0].text
        usage = {
            "prompt_tokens":     response.usage.input_tokens,
            "completion_tokens": response.usage.output_tokens,
            "total_tokens":      response.usage.input_tokens + response.usage.output_tokens,
        }
    else:
        raise ValueError(f"Unknown LLM_PROVIDER: {provider}")

    return text, usage


def parse_action(text: str) -> tuple[str, dict] | None:
    """Extract tool name and arguments from 'Action: tool_name(arg="val")' text."""
    match = re.search(r'Action:\s*(\w+)\((.*)\)', text, re.DOTALL)
    if not match:
        return None
    tool_name = match.group(1)
    args_str  = match.group(2).strip()
    args = {}
    for kv in re.findall(r'(\w+)=["\']?([^"\',)]+)["\']?', args_str):
        args[kv[0]] = kv[1].strip()
    return tool_name, args


def react_agent(query: str, max_iterations: int = 10, verbose: bool = True) -> dict:
    """
    Run a ReAct loop: Thought → Action → Observation → repeat → Final Answer.

    Returns dict with:
      final_answer, trace, iterations, total_time,
      llm_calls, total_tokens, prompt_tokens, completion_tokens,
      estimated_cost_usd, tool_calls
    """
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user",   "content": query},
    ]

    trace              = []
    iterations         = 0
    final_answer       = None
    start_time         = time.perf_counter()
    llm_calls          = 0
    total_prompt_tok   = 0
    total_complete_tok = 0

    while iterations < max_iterations:
        iterations += 1
        if verbose:
            console.print(f"\n[dim]── Iteration {iterations} ──────────────────────────────[/dim]")

        # ── 1. Call LLM ───────────────────────────────────────────────────────
        llm_response, usage = call_llm(messages)
        llm_calls          += 1
        total_prompt_tok   += usage["prompt_tokens"]
        total_complete_tok += usage["completion_tokens"]

        trace.append({"role": "llm", "content": llm_response, "usage": usage})

        if verbose:
            console.print(f"[cyan]{llm_response}[/cyan]")
            console.print(f"[dim]  tokens: {usage['prompt_tokens']} in / {usage['completion_tokens']} out[/dim]")

        # ── 2. Check for Final Answer ─────────────────────────────────────────
        if "Final Answer:" in llm_response:
            final_answer = llm_response.split("Final Answer:", 1)[1].strip()
            break

        # ── 3. Parse and execute Action ───────────────────────────────────────
        parsed = parse_action(llm_response)
        if not parsed:
            if verbose:
                console.print("[yellow]No Action found. Asking model to continue...[/yellow]")
            messages.append({"role": "assistant", "content": llm_response})
            messages.append({"role": "user",      "content": "Continue. Use the Action format."})
            continue

        tool_name, tool_args = parsed

        # ── 4. Execute tool ───────────────────────────────────────────────────
        if tool_name not in TOOL_REGISTRY:
            observation = f"ERROR: Unknown tool '{tool_name}'"
        else:
            try:
                observation = json.dumps(TOOL_REGISTRY[tool_name](**tool_args), indent=2)
            except Exception as e:
                observation = f"ERROR: {e}"

        if verbose:
            console.print(f"[green]Observation: {observation[:300]}{'...' if len(observation) > 300 else ''}[/green]")

        trace.append({"role": "tool", "tool": tool_name, "args": tool_args, "observation": observation})

        # ── 5. Append to conversation ─────────────────────────────────────────
        messages.append({"role": "assistant", "content": llm_response + f"\nObservation: {observation}"})

    total_time     = round(time.perf_counter() - start_time, 3)
    total_tokens   = total_prompt_tok + total_complete_tok
    estimated_cost = estimate_cost(total_prompt_tok, total_complete_tok, LLM_PROVIDER)
    tool_call_count = len([s for s in trace if s["role"] == "tool"])

    return {
        "final_answer":       final_answer,
        "trace":              trace,
        "iterations":         iterations,
        "total_time":         total_time,
        # ── token / cost metrics ──────────────────────────────────────────────
        "llm_calls":          llm_calls,
        "prompt_tokens":      total_prompt_tok,
        "completion_tokens":  total_complete_tok,
        "total_tokens":       total_tokens,
        "estimated_cost_usd": estimated_cost,
        "tool_calls":         tool_call_count,
    }


print("✅ ReAct agent loop ready (token + cost tracking enabled).")

## Run 1 — Happy Path (ACC-001)

Let's sanity-check against the easy case first.

In [ ]:
result_happy = react_agent(
    "Is account ACC-001's package going to arrive on time? Give me a full risk assessment.",
    verbose=True
)

console.print(Panel(
    result_happy["final_answer"],
    title=f"[bold green]Final Answer — ACC-001 (Happy Path)[/bold green]\n"
          f"[dim]{result_happy['iterations']} iterations, {result_happy['total_time']}s[/dim]",
    border_style="green"
))

## Run 2 — The Edge Case (ACC-002)

The same query against the account with two shipments, a disrupted carrier, and a blizzard at the destination.

In [ ]:
result_edge = react_agent(
    "Is account ACC-002's package going to arrive on time? Give me a full risk assessment.",
    verbose=True
)

console.print(Panel(
    result_edge["final_answer"],
    title=f"[bold red]Final Answer — ACC-002 (Edge Case)[/bold red]\n"
          f"[dim]{result_edge['iterations']} iterations, {result_edge['total_time']}s[/dim]",
    border_style="red"
))

## Trace Analysis — What Did the Agent Actually Do?

In [ ]:
from rich.table import Table

table = Table(title="ACC-002 ReAct Decision Trace", show_lines=True)
table.add_column("Step", style="bold", width=6)
table.add_column("Type",  style="cyan",    width=10)
table.add_column("Content",                width=70)

for i, step in enumerate(result_edge["trace"], 1):
    if step["role"] == "llm":
        content = step["content"][:200] + "..." if len(step["content"]) > 200 else step["content"]
        table.add_row(str(i), "[cyan]LLM[/cyan]", content)
    elif step["role"] == "tool":
        obs_preview = step["observation"][:150] + "..." if len(step["observation"]) > 150 else step["observation"]
        table.add_row(str(i), "[green]Tool[/green]", f"{step['tool']}({step['args']})\n→ {obs_preview}")

console.print(table)

In [ ]:
# Summarise the tools called and why
tool_steps = [s for s in result_edge["trace"] if s["role"] == "tool"]
print(f"Total tool calls made: {len(tool_steps)}")
print()
for s in tool_steps:
    print(f"  • {s['tool']}({s['args']})")

## Option B — LangChain ReAct Agent

The same result using LangChain's `create_react_agent` + `@tool` decorators.  
Tools are labelled by their server origin in the docstring so the agent can  
reason about provenance (though it typically ignores this — the point is  
that the tool catalogue is still unified regardless of source).

In [ ]:
from langchain.tools         import tool
from langchain.agents        import create_react_agent, AgentExecutor
from langchain.prompts       import PromptTemplate
from langchain_openai        import ChatOpenAI
from langchain_anthropic     import ChatAnthropic

# ── SERVER A: logistics-internal tools ───────────────────────────────────────

@tool
def lc_get_account(account_id: str) -> str:
    """[logistics-internal] Look up a customer account by ID. Returns name, tier, home_city."""
    return json.dumps(requests.get(f"{LOGISTICS}/accounts/{account_id}").json())

@tool
def lc_get_shipments(account_id: str) -> str:
    """[logistics-internal] Get ALL shipments for an account. Returns a list — there may be more than one!"""
    return json.dumps(requests.get(f"{LOGISTICS}/shipments", params={"account_id": account_id}).json())

@tool
def lc_get_carrier_status(carrier_id: str) -> str:
    """[logistics-internal] Carrier status: on_time/delayed/disrupted. Check delay_hours and disruption_note."""
    return json.dumps(requests.get(f"{LOGISTICS}/carriers/{carrier_id}").json())

# ── SERVER B: weather-provider tools ─────────────────────────────────────────

@tool
def lc_get_weather(city: str) -> str:
    """[weather-provider] Weather conditions + risk_level (low/medium/high) for a city."""
    return json.dumps(requests.get(f"{WEATHER}/weather/{city}").json())

@tool
def lc_get_weather_multi(cities: str) -> str:
    """[weather-provider] Weather for multiple cities. Pass a comma-separated list of city names."""
    city_list = [c.strip() for c in cities.split(",")]
    return json.dumps({city: requests.get(f"{WEATHER}/weather/{city}").json() for city in city_list})

# ── Unified LangChain tool catalogue ─────────────────────────────────────────
lc_tools = [lc_get_account, lc_get_shipments, lc_get_carrier_status,
            lc_get_weather, lc_get_weather_multi]

print(f"LangChain tool catalogue: {[t.name for t in lc_tools]}")
print("  3 from logistics-internal, 2 from weather-provider")

# ── LLM ───────────────────────────────────────────────────────────────────────
if LLM_PROVIDER == "anthropic":
    llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)
else:
    llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ── ReAct prompt ──────────────────────────────────────────────────────────────
react_prompt = PromptTemplate.from_template("""
Answer the following question using the available tools.
Tools come from two sources (logistics-internal and weather-provider) but
you should treat them as a single unified catalogue.
Accounts can have MULTIPLE shipments — process ALL of them.
Check weather for both origin and destination of each shipment.
Cross-reference carrier status with weather risk.

Tools: {tools}
Tool names: {tool_names}

Use this format:
Question: the input question
Thought: what to do
Action: the tool name
Action Input: the input to the tool
Observation: tool result
... (repeat as needed)
Thought: I have enough information
Final Answer: complete risk assessment

Question: {input}
Thought: {agent_scratchpad}
""")

In [ ]:
# Run the LangChain version against the edge case
lc_result = lc_executor.invoke({
    "input": "Is account ACC-002's package going to arrive on time? Full risk assessment please."
})

console.print(Panel(
    lc_result["output"],
    title="[bold blue]LangChain ReAct — ACC-002 Final Answer[/bold blue]",
    border_style="blue"
))

## Save Metrics for Notebook 03

In [ ]:
import json

def result_to_metrics(result: dict) -> dict:
    """Extract all measurable fields from a react_agent result."""
    return {
        "iterations":         result["iterations"],
        "total_time_s":       result["total_time"],
        "llm_calls":          result["llm_calls"],
        "tool_calls":         result["tool_calls"],
        "prompt_tokens":      result["prompt_tokens"],
        "completion_tokens":  result["completion_tokens"],
        "total_tokens":       result["total_tokens"],
        "estimated_cost_usd": result["estimated_cost_usd"],
        "llm_provider":       LLM_PROVIDER,
        "llm_model":          TOKEN_PRICING[LLM_PROVIDER]["model"],
    }

mcp_metrics = {
    "react_happy_path":               result_to_metrics(result_happy),
    "react_edge_case":                result_to_metrics(result_edge),
    "handles_edge_case":              True,
    "has_reasoning":                  True,
    "has_natural_language_explanation": True,
}

with open("../mcp_react_metrics.json", "w") as f:
    json.dump(mcp_metrics, f, indent=2)

# Also save the full traces for the graph in NB03
with open("../react_trace_edge_case.json", "w") as f:
    json.dump(result_edge["trace"], f, indent=2)

print("✅ Metrics and traces saved.")
print()
print("── Happy Path ──────────────────────────────────")
for k, v in mcp_metrics["react_happy_path"].items():
    print(f"  {k:30s} {v}")
print()
print("── Edge Case ───────────────────────────────────")
for k, v in mcp_metrics["react_edge_case"].items():
    print(f"  {k:30s} {v}")

## Summary

Where the strict API chain **silently dropped** the second shipment, the ReAct agent:
1. Read the full shipments list and **decided on its own** to check both
2. Discovered the carrier disruption and **cross-referenced** with Denver weather
3. **Backed up** to check origin weather when destination risk was ambiguous
4. Generated a coherent, prioritised, natural-language risk assessment

No developer wrote `if len(shipments) == 2`. The model reasoned to that conclusion.

---
**Next → `03_analysis_and_blog_prep.ipynb`**